In [1]:
# This cell is removed with the tag: "remove-input"
# As such, it will not be shown in documentation

import warnings
warnings.filterwarnings('ignore')


(Tutorial_Get_charge)=
# Get charge

*Determining formal, tabulated, and force field partial charges.*

The function {func}`molsysmt.physchem.get_charge` evaluates electric charges across hierarchical elements using three distinct definitions:
- `'physical_pH7'`: Standard integer residue charges at neutral pH.
- `'collantes'`: Tabulated residue charges from Collantes et al.
- `'OpenMM'`: Partial atomic charges extracted from molecular mechanics force fields.

:::{versionadded} 1.0.0
:::

:::{admonition} API documentation
:class: dropdown

Follow this link for a detailed description of the input arguments, raised errors, and returned objects of this function: {func}`molsysmt.physchem.get_charge`.
:::


The optional `definition='collantes'` route reproduces electronic charge index
(ECI) scores from the Collantes–Dunn QSAR work. Its legacy `e` wrapper and
higher-level sums are under scientific review in
[#475](https://github.com/uibcdf/molsysmt/issues/475); they do not independently
establish net molecular charge or a force-field assignment. The `physical_pH7`
and `OpenMM` definitions are separate routes.

## Basic usage

By default, `definition='physical_pH7'` evaluates standard integer charges for amino acid residues at physiological pH (e.g. Arg/Lys = +1, Asp/Glu = -1, neutral = 0):


In [2]:
import molsysmt as msm
import pyunitwizard as puw


In [3]:
molsys = msm.convert(msm.systems['Trp-Cage']['1l2y.h5msm'])


:::{note} Demo Systems Catalog
:class: dropdown

This tutorial uses demonstration datasets provided by MolSysMT. To explore the full catalog of bundled systems, forms, and file paths, visit the {ref}`Demo Systems <user-foundations-entrance-demo-systems>` guide.
:::


We query residue charges and the total net charge of the system:


In [4]:
charges_res = msm.physchem.get_charge(molsys, element='group', definition='physical_pH7')
charge_net = msm.physchem.get_charge(molsys, element='system', definition='physical_pH7')

print('Residue charges (elementary charge):', charges_res)
print('Total net system charge:', charge_net)


Residue charges (elementary charge): [0.0 0.0 0.0 0.0 0.0 0.0 0.0 1.0 -1.0 0.0 0.0 0.0 0.0 0.0 0.0 1.0 0.0 0.0 0.0 0.0] elementary_charge
Total net system charge: 1.0 elementary_charge


## Alternative tabulated charges

The `definition='collantes'` parameter applies the continuous tabulated charge scale from Collantes et al.:


In [5]:
charges_collantes = msm.physchem.get_charge(molsys, element='group', definition='collantes')
print('Collantes scale charges:', charges_collantes)


Collantes scale charges: [1.36 0.1 0.72 0.09 1.31 1.08 0.1 0.53 1.25 0.02 0.02 0.16 0.56 0.56 0.02 1.69 0.16 0.16 0.16 0.56] elementary_charge


## Partial atomic charges from force fields

When using `definition='OpenMM'`, partial charges assigned to individual atoms by force field parameter sets (such as AMBER or CHARMM) are extracted:


In [6]:
prmtop_sys = msm.convert(msm.systems['pentalanine']['pentalanine.prmtop'])
omm_sys = msm.convert(prmtop_sys, to_form='openmm.System')
partial_charges = msm.physchem.get_charge(omm_sys, element='atom', definition='OpenMM')
print('First 10 partial atomic charges (e):\n', partial_charges[:10])


First 10 partial atomic charges (e):
 [0.1123 -0.3662 0.1123 0.1123 0.5972 -0.5679 -0.4157 0.2719 0.0337 0.0823] elementary_charge


## Selecting the intended population

This example explicitly uses the optional MDTraj selection syntax. The selected
indices refer to the `group` axis; they are positions, not IDs. Numeric lists
keep their order and repetitions, and an empty supported population keeps a
well-defined empty result. The ordinary MolSysMT syntax remains the default.

Tabulated definitions support groups and their documented aggregates; per-atom
mechanical charges use a different definition. A fixed reference score is not a
stored formal charge or a new partial-charge calculation.

Unknown non-placeholder group names still raise their table diagnostic. These
scales do not validate a group's chemical state or prepare a molecular system.

For a combined example, see {ref}`cookbook-assigning-partial-charges` and the
population notes in {ref}`user-foundations-support-physchem`.


In [7]:
import numpy as np

indices = msm.select(molsys, element='group', selection='resname GLY', syntax='MDTraj')
selected = msm.physchem.get_charge(molsys, element='group',
    selection='resname GLY', syntax='MDTraj')
control = msm.physchem.get_charge(molsys, element='group', selection=indices)
np.testing.assert_allclose(msm.pyunitwizard.get_value(selected, to_unit='e'),
    msm.pyunitwizard.get_value(control, to_unit='e'))
assert len(selected) == len(indices)


:::{seealso} Related Tools & References
:class: dropdown

- {ref}`Convert <Tutorial_Convert>`: Convert molecular systems with {func}`molsysmt.basic.convert`.
- {ref}`Get mass <Tutorial_Get_mass>`: Calculate system mass with {func}`molsysmt.physchem.get_mass`.
- {ref}`Get electronegativity <Tutorial_Get_electronegativity>`: Query atomic electronegativity with {func}`molsysmt.physchem.get_electronegativity`.
:::
